# Germany and EU Tourism Competitiveness Analysis

## 01 Data Mining and Collection

### Purpose

This notebook identifies, evaluates, and collects the data required to analyze Germany's tourism competitiveness within the European Union.

The data collection process is guided by the business objectives and research questions defined in `00_business_understanding.ipynb`.

The primary goals are to:

- collect comparable tourism performance data for Germany and EU member states
- obtain annual and monthly tourism indicators
- identify international source markets
- collect data required for seasonality analysis
- obtain comparable tourism economic indicators where available
- collect 2026 year to date data for comparison with equivalent periods in previous years
- document data definitions, units, geographic coverage, time periods, and source information
- preserve original source data in `data/raw/`

Only data that are sufficiently documented and relevant to the business questions will be included in the analytical workflow.

## 1. Data Requirements

The required data are organized according to the project's research questions.

| Research Area | Required Variables | Preferred Frequency |
|---|---|---|
| Tourism performance | arrivals, overnight stays | Annual |
| EU benchmarking | arrivals, overnight stays, growth | Annual |
| Competitive gap | country, year, tourism performance | Annual |
| Source markets | country of residence, arrivals or nights | Annual / Monthly |
| Source market growth | source country, period, tourism demand | Annual |
| Seasonality | arrivals and overnight stays | Monthly |
| Economic performance | tourism receipts, expenditure | Annual |
| Statistical analysis | selected comparable tourism and economic indicators | Annual |
| 2026 YTD | arrivals and overnight stays | Monthly |
| Strategic analysis | derived rankings, gaps, growth and shares | Derived |

## 2. Source Selection Principles

Data sources will be selected according to the following hierarchy:

1. Official European statistical sources
2. Official national statistical sources
3. International organizations and recognized public institutions
4. Other sources only when the required indicator is unavailable from a higher priority source

For each dataset, the following information will be documented:

- dataset name
- dataset code or identifier
- source organization
- source URL
- collection date
- geographic coverage
- time coverage
- frequency
- unit
- measurement definition
- data status
- comparability notes

Data from different sources will not be combined solely because the variable names appear similar. Definitions, units, periods, and geographic coverage must first be checked for comparability.

## 3. Primary Data Source

### Eurostat

Eurostat will serve as the primary source for comparable European tourism statistics.

The required Eurostat data will be investigated for:

- arrivals at tourist accommodation establishments
- nights spent at tourist accommodation establishments
- resident and non-resident tourism
- country of residence of international visitors
- monthly tourism activity
- annual tourism activity
- tourism expenditure or related economic indicators where available

Before extraction, each Eurostat dataset will be checked for:

- indicator definition
- unit of measurement
- reporting frequency
- geographic coverage
- country of residence classification
- accommodation coverage
- available years
- 2026 data availability
- provisional or estimated observations

## 4. Data Collection Strategy

The collection process will follow four stages.

### Stage 1: Dataset Discovery

Identify the official datasets that correspond to each research question.

### Stage 2: Metadata Validation

Verify definitions, units, frequencies, geographic coverage, time coverage, and comparability before downloading data.

### Stage 3: Raw Data Collection

Retrieve the required observations and preserve the original data without analytical transformations.

Raw files will be stored in:

`data/raw/`

### Stage 4: Collection Validation

Check:

- file dimensions
- available countries
- available years
- available months
- units
- missing observations
- duplicate records
- Germany coverage
- EU coverage
- 2026 availability

Cleaning and analytical transformations will **not** be performed in this notebook. They belong in `02_data_cleaning.ipynb`.

## 5. Preliminary Data Source Inventory

The project will primarily use harmonized European tourism statistics from Eurostat.

| Data Area | Primary Source | Frequency | Main Purpose |
|---|---|---|---|
| Tourist accommodation arrivals | Eurostat | Monthly / Annual | Tourism demand and EU benchmarking |
| Tourist accommodation nights | Eurostat | Monthly / Annual | Tourism performance and benchmarking |
| Resident vs non-resident tourism | Eurostat | Monthly / Annual | International tourism analysis |
| Country of residence | Eurostat | Monthly / Annual | International source market analysis |
| Monthly tourism activity | Eurostat | Monthly | Seasonality and 2026 YTD analysis |
| Accommodation capacity | Eurostat | Annual | Tourism supply and capacity analysis |
| Tourism expenditure | Eurostat | Annual | Economic analysis |
| Additional economic indicators | Eurostat / official EU sources | Annual | Economic context and benchmarking |

### Important Measurement Distinction

Tourist accommodation statistics and tourism demand statistics represent different statistical perspectives.

Accommodation statistics measure activity recorded by tourist accommodation establishments, including arrivals and nights spent.

Tourism demand statistics describe trips made by residents and may include information on trips, nights, destinations, and expenditure.

These datasets will not be treated as interchangeable. Their definitions and analytical roles will be documented before they are combined or compared.

## 6. Data Collection Metadata

A collection log will be maintained for every dataset used in the project.

The following metadata will be recorded:

- dataset name
- Eurostat dataset code
- source organization
- source URL
- date collected
- geographic coverage
- time coverage
- frequency
- unit of measurement
- tourism indicator
- residence classification
- accommodation classification
- provisional or estimated status
- intended analytical use
- comparability notes

This metadata will provide an audit trail from the original source through cleaning, analysis, SQL, and final visualization.

### Data Collection Method

The project prioritizes official structured data access over webpage scraping.

Where available, Eurostat data will be collected through:

1. Eurostat API endpoints
2. Official downloadable CSV or TSV files
3. Web scraping only when no structured data source is available

This approach improves reproducibility, reduces dependence on webpage layout, and preserves the original statistical dimensions and metadata.

In [2]:
import pandas as pd
import numpy as np
import requests
import json

from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
from pathlib import Path

project_root = Path.cwd().parent

raw_path = project_root / "data" / "raw"
clean_path = project_root / "data" / "clean"
processed_path = project_root / "data" / "processed"

raw_path.mkdir(parents=True, exist_ok=True)
clean_path.mkdir(parents=True, exist_ok=True)
processed_path.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data:", raw_path)
print("Clean data:", clean_path)
print("Processed data:", processed_path)

Project root: /Users/jannoelvero/Documents/germany_eu_tourism_analysis
Raw data: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw
Clean data: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/clean
Processed data: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/processed


In [4]:
base_url = (
    "https://ec.europa.eu/eurostat/api/dissemination/"
    "statistics/1.0/data"
)

dataset_code = "tour_occ_nim"

url = f"{base_url}/{dataset_code}"

params = {
    "lang": "en",
    "geo": "DE"
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

print("Status code:", response.status_code)
print("Request URL:", response.url)

Status code: 200
Request URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tour_occ_nim?lang=en&geo=DE


In [5]:
data = response.json()

print("Dataset label:")
print(data.get("label"))

print("\nDimensions:")
print(data.get("id"))

print("\nDimension sizes:")
print(data.get("size"))

Dataset label:
Nights spent at tourist accommodation establishments - monthly data

Dimensions:
['freq', 'c_resid', 'unit', 'nace_r2', 'geo', 'time']

Dimension sizes:
[1, 3, 4, 5, 1, 439]


In [6]:
# Inspect available categories for each dimension
for dimension in ["freq", "c_resid", "unit", "nace_r2", "geo"]:
    
    dim_data = data["dimension"][dimension]["category"]
    
    print(f"\n--- {dimension.upper()} ---")
    
    labels = dim_data.get("label", {})
    
    for code, label in labels.items():
        print(f"{code}: {label}")


--- FREQ ---
M: Monthly

--- C_RESID ---
DOM: Domestic country
FOR: Foreign country
TOTAL: Total

--- UNIT ---
NR: Number
PCH_SM: Percentage change compared to same period in previous year
PCH_SM_2Y: Percentage change compared to same period two years ago
PCH_SM_19: Percentage change compared to same month in 2019

--- NACE_R2 ---
I551-I553: Hotels; holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks
I551: Hotels and similar accommodation
I552_I553: Holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks
I552: Holiday and other short-stay accommodation
I553: Camping grounds, recreational vehicle parks and trailer parks

--- GEO ---
DE: Germany


## 7. Primary International Tourism Measure

For the monthly accommodation analysis, the project defines international tourism demand as:

**Nights spent by foreign residents at tourist accommodation establishments.**

The Eurostat filters are:

- `freq = M`: Monthly
- `c_resid = FOR`: Foreign country
- `unit = NR`: Number
- `nace_r2 = I551-I553`: Hotels, holiday and other short stay accommodation, camping grounds, recreational vehicle parks and trailer parks
- `geo = DE`: Germany

Using the combined accommodation category provides broader tourism coverage than hotels alone.

Domestic tourism will be analyzed separately where relevant and will not be mixed with international tourism demand.

In [7]:
params_de = {
    "lang": "en",
    "geo": "DE",
    "freq": "M",
    "c_resid": "FOR",
    "unit": "NR",
    "nace_r2": "I551-I553"
}

response_de = requests.get(
    f"{base_url}/{dataset_code}",
    params=params_de,
    timeout=30
)

print("Status code:", response_de.status_code)

data_de = response_de.json()

print("Dataset:", data_de.get("label"))
print("Dimensions:", data_de.get("id"))
print("Dimension sizes:", data_de.get("size"))

Status code: 200
Dataset: Nights spent at tourist accommodation establishments - monthly data
Dimensions: ['freq', 'c_resid', 'unit', 'nace_r2', 'geo', 'time']
Dimension sizes: [1, 1, 1, 1, 1, 439]


In [8]:
time_labels = data_de["dimension"]["time"]["category"]["label"]

time_periods = list(time_labels.keys())

print("Number of monthly periods:", len(time_periods))
print("First 10 periods:", time_periods[:10])
print("Last 20 periods:", time_periods[-20:])

Number of monthly periods: 439
First 10 periods: ['1990-01', '1990-02', '1990-03', '1990-04', '1990-05', '1990-06', '1990-07', '1990-08', '1990-09', '1990-10']
Last 20 periods: ['2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']


In [9]:
values = data_de.get("value", {})

print("Number of observations with values:", len(values))

print("\nFirst observations:")
print(list(values.items())[:10])

print("\nLast observations:")
print(list(values.items())[-10:])

Number of observations with values: 438

First observations:
[('0', 1688489), ('1', 2220433), ('2', 2319263), ('3', 2781341), ('4', 3572456), ('5', 4146322), ('6', 6652571), ('7', 6596378), ('8', 4555794), ('9', 3244883)]

Last observations:
[('428', 7674317), ('429', 7195093), ('430', 5590732), ('431', 6582972), ('432', 4277203), ('433', 5078278), ('434', 5197078), ('435', 6596278), ('436', 7446648), ('437', 7653746)]


In [10]:
# Extract time dimension positions
time_index = data_de["dimension"]["time"]["category"]["index"]

# Convert dictionary index to ordered position mapping if necessary
if isinstance(time_index, dict):
    position_to_time = {
        position: period
        for period, position in time_index.items()
    }
else:
    position_to_time = {
        position: period
        for position, period in enumerate(time_periods)
    }

# Extract values and status flags
values = data_de.get("value", {})
status = data_de.get("status", {})

records = []

for position, period in position_to_time.items():
    
    key = str(position)
    
    records.append({
        "country_code": "DE",
        "country": "Germany",
        "period": period,
        "foreign_nights": values.get(key, np.nan),
        "status": status.get(key, None)
    })

germany_monthly_raw = pd.DataFrame(records)

germany_monthly_raw.head()

,country_code,country,period,foreign_nights,status
0,DE,Germany,1990-01,1688489.0,None
1,DE,Germany,1990-02,2220433.0,None
2,DE,Germany,1990-03,2319263.0,None
3,DE,Germany,1990-04,2781341.0,None
4,DE,Germany,1990-05,3572456.0,None


In [11]:
germany_monthly_project = germany_monthly_raw[
    germany_monthly_raw["period"].str[:4].astype(int).between(2021, 2026)
].copy()

germany_monthly_project["year"] = (
    germany_monthly_project["period"]
    .str[:4]
    .astype(int)
)

germany_monthly_project["month"] = (
    germany_monthly_project["period"]
    .str[5:7]
    .astype(int)
)

print("Shape:", germany_monthly_project.shape)

germany_monthly_project.tail(15)

Shape: (67, 7)


,country_code,country,period,foreign_nights,status,year,month
424,DE,Germany,2025-05,7484744.0,None,2025,5
425,DE,Germany,2025-06,7530964.0,None,2025,6
426,DE,Germany,2025-07,10358796.0,None,2025,7
427,DE,Germany,2025-08,9689151.0,None,2025,8
428,DE,Germany,2025-09,7674317.0,None,2025,9
429,DE,Germany,2025-10,7195093.0,None,2025,10
430,DE,Germany,2025-11,5590732.0,None,2025,11
431,DE,Germany,2025-12,6582972.0,None,2025,12
432,DE,Germany,2026-01,4277203.0,None,2026,1
433,DE,Germany,2026-02,5078278.0,None,2026,2


In [12]:
print("Period:")
print(
    germany_monthly_project["period"].min(),
    "to",
    germany_monthly_project["period"].max()
)

print("\nRows:", len(germany_monthly_project))

print("\nMissing values:")
print(germany_monthly_project.isna().sum())

print("\nRows with missing tourism values:")
display(
    germany_monthly_project[
        germany_monthly_project["foreign_nights"].isna()
    ]
)

Period:
2021-01 to 2026-07

Rows: 67

Missing values:
country_code       0
country            0
period             0
foreign_nights     1
status            67
year               0
month              0
dtype: int64

Rows with missing tourism values:


,country_code,country,period,foreign_nights,status,year,month
438,DE,Germany,2026-07,NaN,None,2026,7


In [13]:
coverage_by_year = (
    germany_monthly_project
    .groupby("year")
    .agg(
        months_available=("foreign_nights", "count"),
        first_month=("period", "min"),
        last_month=("period", "max")
    )
    .reset_index()
)

coverage_by_year

,year,months_available,first_month,last_month
0,2021,12,2021-01,2021-12
1,2022,12,2022-01,2022-12
2,2023,12,2023-01,2023-12
3,2024,12,2024-01,2024-12
4,2025,12,2025-01,2025-12
5,2026,6,2026-01,2026-07


In [14]:
print("Status flag counts:")

print(
    germany_monthly_project["status"]
    .fillna("No flag")
    .value_counts(dropna=False)
)

Status flag counts:
status
No flag    67
Name: count, dtype: int64


### Germany Monthly Data Validation

The initial Eurostat extraction confirms complete monthly foreign overnight stay data for Germany from 2021 through 2025.

For 2026, Eurostat currently lists periods from January through July. However, July 2026 does not yet contain a numeric observation for Germany.

Therefore:

- 2021: 12 available months
- 2022: 12 available months
- 2023: 12 available months
- 2024: 12 available months
- 2025: 12 available months
- 2026: 6 available observations, January through June

The current 2026 year to date analytical window is therefore January through June.

Any 2026 year to date comparison must use the equivalent January through June period from previous years.

The July 2026 period will remain missing rather than being imputed as zero.

No observation level status flags were returned for the selected Germany series.

In [15]:
germany_monthly_raw_file = (
    raw_path / "eurostat_germany_foreign_nights_monthly_raw.csv"
)

germany_monthly_project.to_csv(
    germany_monthly_raw_file,
    index=False
)

print("Saved:", germany_monthly_raw_file)
print("Shape:", germany_monthly_project.shape)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/eurostat_germany_foreign_nights_monthly_raw.csv
Shape: (67, 7)


In [16]:
collection_metadata = pd.DataFrame({
    "dataset_name": [
        "Nights spent at tourist accommodation establishments - monthly data"
    ],
    "dataset_code": [
        "tour_occ_nim"
    ],
    "source": [
        "Eurostat"
    ],
    "geographic_coverage": [
        "Germany"
    ],
    "time_coverage": [
        "2021-01 to 2026-07"
    ],
    "frequency": [
        "Monthly"
    ],
    "unit": [
        "Number of nights"
    ],
    "residence": [
        "Foreign country"
    ],
    "accommodation_scope": [
        "I551-I553"
    ],
    "collection_date": [
        datetime.now().date().isoformat()
    ],
    "analytical_use": [
        "International tourism demand, seasonality, and 2026 YTD analysis"
    ],
    "notes": [
        "2026-07 period available but numeric value missing for Germany. Current valid 2026 YTD window is January-June."
    ]
})

collection_metadata

,dataset_name,dataset_code,source,geographic_coverage,time_coverage,frequency,unit,residence,accommodation_scope,collection_date,analytical_use,notes
0,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"International tourism demand, seasonality, and...",2026-07 period available but numeric value mis...


In [17]:
metadata_file = raw_path / "data_collection_metadata.csv"

collection_metadata.to_csv(
    metadata_file,
    index=False
)

print("Saved:", metadata_file)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/data_collection_metadata.csv


## 8. EU27 Benchmark Data Collection

After validating the Eurostat API using Germany, the same tourism indicator will be collected for all 27 European Union member states.

The benchmark uses the same definition applied to Germany:

- Frequency: Monthly
- Residence: Foreign country
- Unit: Number of nights
- Accommodation scope: I551-I553
- Analytical period: 2021-2026

Maintaining identical definitions across countries is necessary for a valid comparison of Germany's tourism performance.

The EU27 extraction will support:

- Germany's position among EU tourism destinations
- Cross-country tourism demand comparisons
- Growth analysis
- Seasonality comparisons
- 2026 year-to-date benchmarking
- Statistical analysis

Missing observations will remain missing and will not be interpreted as zero.

In [18]:
eu27_countries = {
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "HR": "Croatia",
    "CY": "Cyprus",
    "CZ": "Czechia",
    "DK": "Denmark",
    "EE": "Estonia",
    "FI": "Finland",
    "FR": "France",
    "DE": "Germany",
    "EL": "Greece",
    "HU": "Hungary",
    "IE": "Ireland",
    "IT": "Italy",
    "LV": "Latvia",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "MT": "Malta",
    "NL": "Netherlands",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "SK": "Slovakia",
    "SI": "Slovenia",
    "ES": "Spain",
    "SE": "Sweden"
}

eu27_reference = pd.DataFrame(
    eu27_countries.items(),
    columns=["country_code", "country"]
)

print("Number of EU countries:", len(eu27_reference))

eu27_reference

Number of EU countries: 27


,country_code,country
0,AT,Austria
1,BE,Belgium
2,BG,Bulgaria
3,HR,Croatia
4,CY,Cyprus
5,CZ,Czechia
6,DK,Denmark
7,EE,Estonia
8,FI,Finland
9,FR,France


In [19]:
metadata_response = requests.get(
    f"{base_url}/{dataset_code}",
    params={"lang": "en"},
    timeout=30
)

print("Status code:", metadata_response.status_code)

metadata_data = metadata_response.json()

geo_labels = (
    metadata_data["dimension"]["geo"]["category"]["label"]
)

print("Number of geographic categories:", len(geo_labels))

Status code: 200
Number of geographic categories: 44


In [20]:
available_geo_codes = set(geo_labels.keys())
eu27_codes = set(eu27_countries.keys())

missing_codes = eu27_codes - available_geo_codes

print("EU27 countries expected:", len(eu27_codes))
print("EU27 codes found in Eurostat:", len(eu27_codes & available_geo_codes))
print("Missing codes:", missing_codes)

EU27 countries expected: 27
EU27 codes found in Eurostat: 27
Missing codes: set()


In [21]:
eu27_records = []

for country_code, country_name in eu27_countries.items():
    
    params_country = {
        "lang": "en",
        "geo": country_code,
        "freq": "M",
        "c_resid": "FOR",
        "unit": "NR",
        "nace_r2": "I551-I553"
    }
    
    response_country = requests.get(
        f"{base_url}/{dataset_code}",
        params=params_country,
        timeout=30
    )
    
    if response_country.status_code != 200:
        print(
            f"Failed: {country_name} "
            f"({response_country.status_code})"
        )
        continue
    
    country_data = response_country.json()
    
    time_index = (
        country_data["dimension"]["time"]
        ["category"]["index"]
    )
    
    values = country_data.get("value", {})
    status = country_data.get("status", {})
    
    for period, position in time_index.items():
        
        year = int(period[:4])
        
        if 2021 <= year <= 2026:
            
            key = str(position)
            
            eu27_records.append({
                "country_code": country_code,
                "country": country_name,
                "period": period,
                "foreign_nights": values.get(key, np.nan),
                "status": status.get(key, None),
                "year": year,
                "month": int(period[5:7])
            })

eu27_monthly_raw = pd.DataFrame(eu27_records)

print("Extraction complete.")
print("Shape:", eu27_monthly_raw.shape)
print("Countries:", eu27_monthly_raw["country_code"].nunique())

eu27_monthly_raw.head()

Extraction complete.
Shape: (1809, 7)
Countries: 27


,country_code,country,period,foreign_nights,status,year,month
0,AT,Austria,2021-01,148315.0,NaN,2021,1
1,AT,Austria,2021-02,172759.0,NaN,2021,2
2,AT,Austria,2021-03,207381.0,NaN,2021,3
3,AT,Austria,2021-04,200806.0,NaN,2021,4
4,AT,Austria,2021-05,994223.0,NaN,2021,5


In [22]:
country_coverage = (
    eu27_monthly_raw
    .groupby(["country_code", "country"])
    .agg(
        rows=("period", "size"),
        observations=("foreign_nights", "count"),
        missing_values=("foreign_nights", lambda x: x.isna().sum()),
        first_period=("period", "min"),
        last_period=("period", "max")
    )
    .reset_index()
)

print("Countries:", len(country_coverage))
print(
    "Total missing tourism observations:",
    country_coverage["missing_values"].sum()
)

country_coverage

Countries: 27
Total missing tourism observations: 51


,country_code,country,rows,observations,missing_values,first_period,last_period
0,AT,Austria,67,66,1,2021-01,2026-07
1,BE,Belgium,67,66,1,2021-01,2026-07
2,BG,Bulgaria,67,47,20,2021-01,2026-07
3,CY,Cyprus,67,66,1,2021-01,2026-07
4,CZ,Czechia,67,66,1,2021-01,2026-07
5,DE,Germany,67,66,1,2021-01,2026-07
6,DK,Denmark,67,66,1,2021-01,2026-07
7,EE,Estonia,67,59,8,2021-01,2026-07
8,EL,Greece,67,66,1,2021-01,2026-07
9,ES,Spain,67,66,1,2021-01,2026-07


In [23]:
coverage_2026 = (
    eu27_monthly_raw[
        eu27_monthly_raw["year"] == 2026
    ]
    .groupby(["country_code", "country"])
    .agg(
        periods_listed=("period", "size"),
        months_with_data=("foreign_nights", "count"),
        missing_values=("foreign_nights", lambda x: x.isna().sum()),
        latest_period_listed=("period", "max")
    )
    .reset_index()
)

coverage_2026.sort_values(
    ["months_with_data", "country"],
    ascending=[True, True]
)

,country_code,country,periods_listed,months_with_data,missing_values,latest_period_listed
0,AT,Austria,7,6,1,2026-07
1,BE,Belgium,7,6,1,2026-07
2,BG,Bulgaria,7,6,1,2026-07
12,HR,Croatia,7,6,1,2026-07
3,CY,Cyprus,7,6,1,2026-07
4,CZ,Czechia,7,6,1,2026-07
6,DK,Denmark,7,6,1,2026-07
7,EE,Estonia,7,6,1,2026-07
11,FR,France,7,6,1,2026-07
5,DE,Germany,7,6,1,2026-07


In [24]:
duplicates = (
    eu27_monthly_raw
    .duplicated(
        subset=["country_code", "period"],
        keep=False
    )
    .sum()
)

print("Duplicate country-period rows:", duplicates)

Duplicate country-period rows: 0


In [25]:
missing_observations = (
    eu27_monthly_raw[
        eu27_monthly_raw["foreign_nights"].isna()
    ]
    .sort_values(
        ["country_code", "period"]
    )
)

print(
    "Total missing observations:",
    len(missing_observations)
)

missing_observations

Total missing observations: 51


,country_code,country,period,foreign_nights,status,year,month
66,AT,Austria,2026-07,NaN,NaN,2026,7
133,BE,Belgium,2026-07,NaN,NaN,2026,7
134,BG,Bulgaria,2021-01,NaN,|C,2021,1
135,BG,Bulgaria,2021-02,NaN,|C,2021,2
136,BG,Bulgaria,2021-03,NaN,|C,2021,3
137,BG,Bulgaria,2021-04,NaN,|C,2021,4
138,BG,Bulgaria,2021-05,NaN,|C,2021,5
143,BG,Bulgaria,2021-10,NaN,|C,2021,10
146,BG,Bulgaria,2022-01,NaN,|C,2022,1
147,BG,Bulgaria,2022-02,NaN,|C,2022,2


In [26]:
missing_by_country_year = (
    missing_observations
    .groupby(
        ["country_code", "country", "year"]
    )
    .size()
    .reset_index(name="missing_months")
)

missing_by_country_year

,country_code,country,year,missing_months
0,AT,Austria,2026,1
1,BE,Belgium,2026,1
2,BG,Bulgaria,2021,6
3,BG,Bulgaria,2022,6
4,BG,Bulgaria,2023,7
5,BG,Bulgaria,2026,1
6,CY,Cyprus,2026,1
7,CZ,Czechia,2026,1
8,DE,Germany,2026,1
9,DK,Denmark,2026,1


In [27]:
missing_observations[
    missing_observations["country_code"].isin(
        ["BG", "EE"]
    )
]

,country_code,country,period,foreign_nights,status,year,month
134,BG,Bulgaria,2021-01,NaN,|C,2021,1
135,BG,Bulgaria,2021-02,NaN,|C,2021,2
136,BG,Bulgaria,2021-03,NaN,|C,2021,3
137,BG,Bulgaria,2021-04,NaN,|C,2021,4
138,BG,Bulgaria,2021-05,NaN,|C,2021,5
143,BG,Bulgaria,2021-10,NaN,|C,2021,10
146,BG,Bulgaria,2022-01,NaN,|C,2022,1
147,BG,Bulgaria,2022-02,NaN,|C,2022,2
148,BG,Bulgaria,2022-03,NaN,|C,2022,3
155,BG,Bulgaria,2022-10,NaN,|C,2022,10


### EU27 Missing Data Validation

The EU27 monthly extraction contains 51 observations without numeric tourism values.

Two different forms of missingness were identified:

1. **Current-period availability**
   - July 2026 is listed but is not yet numerically available for most EU countries.
   - Finland and Slovenia currently contain July 2026 observations.
   - To maintain comparability, the current common EU27 year-to-date window will be January through June 2026.

2. **Historical flagged observations**
   - Bulgaria contains multiple missing observations during 2021-2023.
   - Estonia contains several missing observations during 2021-2023.
   - These observations contain a Eurostat status flag (`C`) and therefore should not be treated as ordinary missing values without investigating the flag definition.

No missing tourism observation will be automatically replaced with zero or statistically imputed.

The raw extraction will preserve all values and status flags exactly as returned. Decisions concerning historical flagged observations will be documented during data cleaning and analytical validation.

In [28]:
eu27_raw_file = (
    raw_path /
    "eurostat_eu27_foreign_nights_monthly_raw.csv"
)

eu27_monthly_raw.to_csv(
    eu27_raw_file,
    index=False
)

print("Saved:", eu27_raw_file)
print("Shape:", eu27_monthly_raw.shape)
print(
    "Countries:",
    eu27_monthly_raw["country_code"].nunique()
)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/eurostat_eu27_foreign_nights_monthly_raw.csv
Shape: (1809, 7)
Countries: 27


In [29]:
eu27_reference_file = (
    raw_path /
    "eu27_country_reference.csv"
)

eu27_reference.to_csv(
    eu27_reference_file,
    index=False
)

print("Saved:", eu27_reference_file)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/eu27_country_reference.csv


In [30]:
eu27_metadata = pd.DataFrame({
    "dataset_name": [
        "Nights spent at tourist accommodation establishments - monthly data"
    ],
    "dataset_code": [
        "tour_occ_nim"
    ],
    "source": [
        "Eurostat"
    ],
    "geographic_coverage": [
        "EU27 member states"
    ],
    "time_coverage": [
        "2021-01 to 2026-07"
    ],
    "frequency": [
        "Monthly"
    ],
    "unit": [
        "Number of nights"
    ],
    "residence": [
        "Foreign country"
    ],
    "accommodation_scope": [
        "I551-I553"
    ],
    "collection_date": [
        datetime.now().date().isoformat()
    ],
    "analytical_use": [
        "EU benchmarking, growth, seasonality, statistics, and 2026 YTD analysis"
    ],
    "notes": [
        "Common 2026 EU27 comparison window is January-June. Historical flagged missing observations identified for Bulgaria and Estonia."
    ]
})

collection_metadata = pd.concat(
    [collection_metadata, eu27_metadata],
    ignore_index=True
)

collection_metadata.to_csv(
    metadata_file,
    index=False
)

collection_metadata

,dataset_name,dataset_code,source,geographic_coverage,time_coverage,frequency,unit,residence,accommodation_scope,collection_date,analytical_use,notes
0,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"International tourism demand, seasonality, and...",2026-07 period available but numeric value mis...
1,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"EU benchmarking, growth, seasonality, statisti...",Common 2026 EU27 comparison window is January-...


### Eurostat Confidentiality Flag

Some historical observations for Bulgaria and Estonia contain the Eurostat status flag `C`.

Eurostat uses `C` to indicate confidential statistical information. In these cases, the underlying value exists but is not publicly disclosed.

For this project:

- confidential observations will remain missing
- confidential observations will not be replaced with zero
- confidential observations will not be statistically imputed
- annual totals that depend on confidential monthly observations will not be interpreted as complete annual totals
- affected country-year combinations will be identified during data cleaning and excluded from analyses requiring complete annual observations where necessary

This preserves the integrity of the official source and prevents artificial distortion of country rankings, growth rates, and statistical results.

## 9. Tourist Accommodation Arrivals

The second core tourism indicator is the number of arrivals at tourist accommodation establishments.

Eurostat dataset:

**`tour_occ_arm` — Arrivals at tourist accommodation establishments - monthly data**

Arrivals complement overnight stays by measuring the number of guests checking into accommodation establishments, while nights spent measure the duration of their stays.

Analyzing both indicators will allow the project to distinguish between:

- tourism volume
- visitor frequency
- length of stay
- differences between Germany and competing EU destinations

The same residence and accommodation classifications used for overnight stays will be applied where available to maintain comparability.

In [31]:
arrivals_dataset_code = "tour_occ_arm"

arrivals_url = f"{base_url}/{arrivals_dataset_code}"

arrivals_params_de = {
    "lang": "en",
    "geo": "DE"
}

arrivals_response_de = requests.get(
    arrivals_url,
    params=arrivals_params_de,
    timeout=30
)

print("Status code:", arrivals_response_de.status_code)
print("Request URL:", arrivals_response_de.url)

Status code: 200
Request URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tour_occ_arm?lang=en&geo=DE


In [32]:
arrivals_data_de = arrivals_response_de.json()

print("Dataset label:")
print(arrivals_data_de.get("label"))

print("\nDimensions:")
print(arrivals_data_de.get("id"))

print("\nDimension sizes:")
print(arrivals_data_de.get("size"))

Dataset label:
Arrivals at tourist accommodation establishments - monthly data

Dimensions:
['freq', 'c_resid', 'unit', 'nace_r2', 'geo', 'time']

Dimension sizes:
[1, 3, 4, 5, 1, 439]


In [33]:
for dimension in arrivals_data_de.get("id", []):
    
    if dimension == "time":
        continue
    
    dim_data = (
        arrivals_data_de["dimension"][dimension]["category"]
    )
    
    print(f"\n--- {dimension.upper()} ---")
    
    labels = dim_data.get("label", {})
    
    for code, label in labels.items():
        print(f"{code}: {label}")


--- FREQ ---
M: Monthly

--- C_RESID ---
DOM: Domestic country
FOR: Foreign country
TOTAL: Total

--- UNIT ---
NR: Number
PCH_SM: Percentage change compared to same period in previous year
PCH_SM_2Y: Percentage change compared to same period two years ago
PCH_SM_19: Percentage change compared to same month in 2019

--- NACE_R2 ---
I551-I553: Hotels; holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks
I551: Hotels and similar accommodation
I552_I553: Holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks
I552: Holiday and other short-stay accommodation
I553: Camping grounds, recreational vehicle parks and trailer parks

--- GEO ---
DE: Germany


In [34]:
arrivals_params_de = {
    "lang": "en",
    "geo": "DE",
    "freq": "M",
    "c_resid": "FOR",
    "unit": "NR",
    "nace_r2": "I551-I553"
}

arrivals_response_de = requests.get(
    arrivals_url,
    params=arrivals_params_de,
    timeout=30
)

print("Status code:", arrivals_response_de.status_code)

arrivals_filtered_de = arrivals_response_de.json()

print("Dataset:", arrivals_filtered_de.get("label"))
print("Dimensions:", arrivals_filtered_de.get("id"))
print("Dimension sizes:", arrivals_filtered_de.get("size"))

Status code: 200
Dataset: Arrivals at tourist accommodation establishments - monthly data
Dimensions: ['freq', 'c_resid', 'unit', 'nace_r2', 'geo', 'time']
Dimension sizes: [1, 1, 1, 1, 1, 439]


In [35]:
time_index = (
    arrivals_filtered_de["dimension"]["time"]
    ["category"]["index"]
)

values = arrivals_filtered_de.get("value", {})
status = arrivals_filtered_de.get("status", {})

arrival_records = []

for period, position in time_index.items():
    
    year = int(period[:4])
    
    if 2021 <= year <= 2026:
        
        key = str(position)
        
        arrival_records.append({
            "country_code": "DE",
            "country": "Germany",
            "period": period,
            "foreign_arrivals": values.get(key, np.nan),
            "status": status.get(key, None),
            "year": year,
            "month": int(period[5:7])
        })

germany_arrivals_raw = pd.DataFrame(arrival_records)

print("Shape:", germany_arrivals_raw.shape)

germany_arrivals_raw.tail(15)

Shape: (67, 7)


,country_code,country,period,foreign_arrivals,status,year,month
52,DE,Germany,2025-05,3329016.0,None,2025,5
53,DE,Germany,2025-06,3447435.0,None,2025,6
54,DE,Germany,2025-07,4666150.0,None,2025,7
55,DE,Germany,2025-08,4229612.0,None,2025,8
56,DE,Germany,2025-09,3451143.0,None,2025,9
57,DE,Germany,2025-10,3171850.0,None,2025,10
58,DE,Germany,2025-11,2550385.0,None,2025,11
59,DE,Germany,2025-12,3045185.0,None,2025,12
60,DE,Germany,2026-01,1938956.0,None,2026,1
61,DE,Germany,2026-02,2271693.0,None,2026,2


In [36]:
arrivals_coverage = (
    germany_arrivals_raw
    .groupby("year")
    .agg(
        months_available=("foreign_arrivals", "count"),
        missing_values=(
            "foreign_arrivals",
            lambda x: x.isna().sum()
        ),
        first_period=("period", "min"),
        last_period=("period", "max")
    )
    .reset_index()
)

arrivals_coverage

,year,months_available,missing_values,first_period,last_period
0,2021,12,0,2021-01,2021-12
1,2022,12,0,2022-01,2022-12
2,2023,12,0,2023-01,2023-12
3,2024,12,0,2024-01,2024-12
4,2025,12,0,2025-01,2025-12
5,2026,6,1,2026-01,2026-07


In [37]:
germany_arrivals_missing = germany_arrivals_raw[
    germany_arrivals_raw["foreign_arrivals"].isna()
]

print(
    "Missing arrival observations:",
    len(germany_arrivals_missing)
)

germany_arrivals_missing

Missing arrival observations: 1


,country_code,country,period,foreign_arrivals,status,year,month
66,DE,Germany,2026-07,NaN,None,2026,7


### Germany Arrivals Data Validation

The Germany foreign arrivals series is complete for 2021 through 2025.

For 2026, numeric observations are currently available from January through June. July 2026 is listed by Eurostat but does not yet contain a numeric value.

Therefore, the current Germany year-to-date comparison period is January through June 2026.

This coverage is consistent with the foreign overnight stays series, allowing both indicators to be compared over identical periods.

In [38]:
germany_arrivals_file = (
    raw_path /
    "eurostat_germany_foreign_arrivals_monthly_raw.csv"
)

germany_arrivals_raw.to_csv(
    germany_arrivals_file,
    index=False
)

print("Saved:", germany_arrivals_file)
print("Shape:", germany_arrivals_raw.shape)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/eurostat_germany_foreign_arrivals_monthly_raw.csv
Shape: (67, 7)


In [39]:
eu27_arrival_records = []

for country_code, country_name in eu27_countries.items():
    
    params_country = {
        "lang": "en",
        "geo": country_code,
        "freq": "M",
        "c_resid": "FOR",
        "unit": "NR",
        "nace_r2": "I551-I553"
    }
    
    response_country = requests.get(
        arrivals_url,
        params=params_country,
        timeout=30
    )
    
    if response_country.status_code != 200:
        print(
            f"Failed: {country_name} "
            f"({response_country.status_code})"
        )
        continue
    
    country_data = response_country.json()
    
    time_index = (
        country_data["dimension"]["time"]
        ["category"]["index"]
    )
    
    values = country_data.get("value", {})
    status = country_data.get("status", {})
    
    for period, position in time_index.items():
        
        year = int(period[:4])
        
        if 2021 <= year <= 2026:
            
            key = str(position)
            
            eu27_arrival_records.append({
                "country_code": country_code,
                "country": country_name,
                "period": period,
                "foreign_arrivals": values.get(key, np.nan),
                "status": status.get(key, None),
                "year": year,
                "month": int(period[5:7])
            })

eu27_arrivals_raw = pd.DataFrame(
    eu27_arrival_records
)

print("Extraction complete.")
print("Shape:", eu27_arrivals_raw.shape)
print(
    "Countries:",
    eu27_arrivals_raw["country_code"].nunique()
)

eu27_arrivals_raw.head()

Extraction complete.
Shape: (1809, 7)
Countries: 27


,country_code,country,period,foreign_arrivals,status,year,month
0,AT,Austria,2021-01,39058.0,NaN,2021,1
1,AT,Austria,2021-02,43690.0,NaN,2021,2
2,AT,Austria,2021-03,54888.0,NaN,2021,3
3,AT,Austria,2021-04,57536.0,NaN,2021,4
4,AT,Austria,2021-05,312967.0,NaN,2021,5


In [40]:
arrival_country_coverage = (
    eu27_arrivals_raw
    .groupby(["country_code", "country"])
    .agg(
        rows=("period", "size"),
        observations=("foreign_arrivals", "count"),
        missing_values=(
            "foreign_arrivals",
            lambda x: x.isna().sum()
        ),
        first_period=("period", "min"),
        last_period=("period", "max")
    )
    .reset_index()
)

print(
    "Countries:",
    len(arrival_country_coverage)
)

print(
    "Total missing arrival observations:",
    arrival_country_coverage["missing_values"].sum()
)

arrival_country_coverage

Countries: 27
Total missing arrival observations: 57


,country_code,country,rows,observations,missing_values,first_period,last_period
0,AT,Austria,67,66,1,2021-01,2026-07
1,BE,Belgium,67,65,2,2021-01,2026-07
2,BG,Bulgaria,67,48,19,2021-01,2026-07
3,CY,Cyprus,67,66,1,2021-01,2026-07
4,CZ,Czechia,67,66,1,2021-01,2026-07
5,DE,Germany,67,66,1,2021-01,2026-07
6,DK,Denmark,67,64,3,2021-01,2026-07
7,EE,Estonia,67,59,8,2021-01,2026-07
8,EL,Greece,67,66,1,2021-01,2026-07
9,ES,Spain,67,66,1,2021-01,2026-07


In [41]:
arrival_coverage_2026 = (
    eu27_arrivals_raw[
        eu27_arrivals_raw["year"] == 2026
    ]
    .groupby(["country_code", "country"])
    .agg(
        periods_listed=("period", "size"),
        months_with_data=("foreign_arrivals", "count"),
        missing_values=(
            "foreign_arrivals",
            lambda x: x.isna().sum()
        ),
        latest_period_listed=("period", "max")
    )
    .reset_index()
)

arrival_coverage_2026.sort_values(
    ["months_with_data", "country"],
    ascending=[True, True]
)

,country_code,country,periods_listed,months_with_data,missing_values,latest_period_listed
22,PT,Portugal,7,4,3,2026-07
0,AT,Austria,7,6,1,2026-07
1,BE,Belgium,7,6,1,2026-07
2,BG,Bulgaria,7,6,1,2026-07
12,HR,Croatia,7,6,1,2026-07
3,CY,Cyprus,7,6,1,2026-07
4,CZ,Czechia,7,6,1,2026-07
6,DK,Denmark,7,6,1,2026-07
7,EE,Estonia,7,6,1,2026-07
11,FR,France,7,6,1,2026-07


In [42]:
missing_arrivals = (
    eu27_arrivals_raw[
        eu27_arrivals_raw["foreign_arrivals"].isna()
    ]
    .sort_values(
        ["country_code", "period"]
    )
)

print(
    "Total missing arrival observations:",
    len(missing_arrivals)
)

missing_arrivals

Total missing arrival observations: 57


,country_code,country,period,foreign_arrivals,status,year,month
66,AT,Austria,2026-07,NaN,NaN,2026,7
67,BE,Belgium,2021-01,NaN,NaN,2021,1
133,BE,Belgium,2026-07,NaN,NaN,2026,7
134,BG,Bulgaria,2021-01,NaN,|C,2021,1
135,BG,Bulgaria,2021-02,NaN,|C,2021,2
136,BG,Bulgaria,2021-03,NaN,|C,2021,3
137,BG,Bulgaria,2021-04,NaN,|C,2021,4
138,BG,Bulgaria,2021-05,NaN,|C,2021,5
143,BG,Bulgaria,2021-10,NaN,|C,2021,10
146,BG,Bulgaria,2022-01,NaN,|C,2022,1


In [43]:
missing_arrivals_by_year = (
    missing_arrivals
    .groupby(
        ["country_code", "country", "year"]
    )
    .size()
    .reset_index(name="missing_months")
)

missing_arrivals_by_year

,country_code,country,year,missing_months
0,AT,Austria,2026,1
1,BE,Belgium,2021,1
2,BE,Belgium,2026,1
3,BG,Bulgaria,2021,6
4,BG,Bulgaria,2022,6
5,BG,Bulgaria,2023,5
6,BG,Bulgaria,2025,1
7,BG,Bulgaria,2026,1
8,CY,Cyprus,2026,1
9,CZ,Czechia,2026,1


In [44]:
duplicates = (
    eu27_arrivals_raw
    .duplicated(
        subset=["country_code", "period"],
        keep=False
    )
    .sum()
)

print(
    "Duplicate country-period rows:",
    duplicates
)

Duplicate country-period rows: 0


### EU27 Arrivals Data Validation

The EU27 foreign arrivals extraction contains 1,809 country-month records covering 27 EU member states from January 2021 through July 2026.

The dataset contains 57 missing numeric observations.

Missingness is not uniform across countries or periods:

- Most countries currently have foreign arrivals data through June 2026.
- Finland and Slovenia already contain July 2026 observations.
- Portugal currently has only four numeric observations in 2026 because February, March, and July are unavailable.
- Bulgaria and Estonia contain several historical confidential observations.
- Several other countries contain isolated historical missing observations.

No missing arrival observation will be replaced with zero or automatically imputed.

For Germany-specific 2026 analysis, January through June will be used.

For EU27 comparative analysis, country-period completeness will be checked before each calculation. Countries with incomplete observations for the required comparison period will be identified and excluded from analyses requiring complete periods rather than being assigned artificial values.

Analyses combining arrivals and overnight stays, including average length of stay, will use matched country-month observations only.

In [45]:
eu27_arrivals_file = (
    raw_path /
    "eurostat_eu27_foreign_arrivals_monthly_raw.csv"
)

eu27_arrivals_raw.to_csv(
    eu27_arrivals_file,
    index=False
)

print("Saved:", eu27_arrivals_file)
print("Shape:", eu27_arrivals_raw.shape)
print(
    "Countries:",
    eu27_arrivals_raw["country_code"].nunique()
)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/eurostat_eu27_foreign_arrivals_monthly_raw.csv
Shape: (1809, 7)
Countries: 27


In [46]:
arrivals_metadata = pd.DataFrame({
    "dataset_name": [
        "Arrivals at tourist accommodation establishments - monthly data"
    ],
    "dataset_code": [
        "tour_occ_arm"
    ],
    "source": [
        "Eurostat"
    ],
    "geographic_coverage": [
        "EU27 member states"
    ],
    "time_coverage": [
        "2021-01 to 2026-07"
    ],
    "frequency": [
        "Monthly"
    ],
    "unit": [
        "Number of arrivals"
    ],
    "residence": [
        "Foreign country"
    ],
    "accommodation_scope": [
        "I551-I553"
    ],
    "collection_date": [
        datetime.now().date().isoformat()
    ],
    "analytical_use": [
        "EU benchmarking, tourism volume, growth, seasonality, 2026 YTD, and average length of stay"
    ],
    "notes": [
        "Missing observations vary by country. Germany has January-June 2026 available. Portugal has incomplete January-June 2026 arrivals. Comparative analyses require period-completeness checks."
    ]
})

collection_metadata = pd.concat(
    [collection_metadata, arrivals_metadata],
    ignore_index=True
)

collection_metadata.to_csv(
    metadata_file,
    index=False
)

collection_metadata

,dataset_name,dataset_code,source,geographic_coverage,time_coverage,frequency,unit,residence,accommodation_scope,collection_date,analytical_use,notes
0,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"International tourism demand, seasonality, and...",2026-07 period available but numeric value mis...
1,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"EU benchmarking, growth, seasonality, statisti...",Common 2026 EU27 comparison window is January-...
2,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"EU benchmarking, tourism volume, growth, seaso...",Missing observations vary by country. Germany ...


In [47]:
germany_arrivals_metadata = arrivals_metadata.copy()

germany_arrivals_metadata["geographic_coverage"] = "Germany"

germany_arrivals_metadata["analytical_use"] = (
    "Germany tourism performance, seasonality, "
    "2026 YTD, and average length of stay"
)

germany_arrivals_metadata["notes"] = (
    "Complete 2021-2025. January-June 2026 available; "
    "July 2026 listed but numeric value unavailable."
)

collection_metadata = pd.concat(
    [collection_metadata, germany_arrivals_metadata],
    ignore_index=True
)

collection_metadata.to_csv(
    metadata_file,
    index=False
)

collection_metadata

,dataset_name,dataset_code,source,geographic_coverage,time_coverage,frequency,unit,residence,accommodation_scope,collection_date,analytical_use,notes
0,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"International tourism demand, seasonality, and...",2026-07 period available but numeric value mis...
1,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"EU benchmarking, growth, seasonality, statisti...",Common 2026 EU27 comparison window is January-...
2,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"EU benchmarking, tourism volume, growth, seaso...",Missing observations vary by country. Germany ...
3,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"Germany tourism performance, seasonality, 2026...",Complete 2021-2025. January-June 2026 availabl...


## 10. International Source Market Data

Germany's overall foreign arrivals and overnight stays show the scale of international tourism demand but do not identify which countries generate that demand.

Source market analysis will therefore examine tourism activity in Germany by visitors' country of residence.

The analysis will support:

- identification of Germany's largest international source markets
- source market share analysis
- identification of growing and declining markets
- comparison of source market performance over time
- analysis of seasonal demand by source market where data permits
- identification of potential market development opportunities

Country of residence will be used as the source market definition rather than nationality.

Only comparable official Eurostat observations will be used, and aggregate categories will be distinguished from individual source countries.

In [49]:
source_market_dataset_code = "tour_occ_ninraw"

source_market_url = (
    f"{base_url}/{source_market_dataset_code}"
)

source_market_params_de = {
    "lang": "en",
    "geo": "DE"
}

source_market_response = requests.get(
    source_market_url,
    params=source_market_params_de,
    timeout=30
)

print("Status code:", source_market_response.status_code)
print("Request URL:", source_market_response.url)

Status code: 200
Request URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tour_occ_ninraw?lang=en&geo=DE


In [50]:
source_market_data = source_market_response.json()

print("Dataset label:")
print(source_market_data.get("label"))

print("\nDimensions:")
print(source_market_data.get("id"))

print("\nDimension sizes:")
print(source_market_data.get("size"))

Dataset label:
Nights spent at tourist accommodation establishments  by country of origin of the tourist

Dimensions:
['freq', 'unit', 'nace_r2', 'c_resid', 'geo', 'time']

Dimension sizes:
[1, 2, 5, 63, 1, 36]


In [51]:
for dimension in source_market_data.get("id", []):

    if dimension == "time":
        continue

    dim_data = (
        source_market_data["dimension"][dimension]
        ["category"]
    )

    print(f"\n--- {dimension.upper()} ---")

    labels = dim_data.get("label", {})

    for code, label in labels.items():
        print(f"{code}: {label}")


--- FREQ ---
A: Annual

--- UNIT ---
NR: Number
PCH_PRE: Percentage change on previous period

--- NACE_R2 ---
I551-I553: Hotels; holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks
I551: Hotels and similar accommodation
I552_I553: Holiday and other short-stay accommodation; camping grounds, recreational vehicle parks and trailer parks
I552: Holiday and other short-stay accommodation
I553: Camping grounds, recreational vehicle parks and trailer parks

--- C_RESID ---
EUR: Europe
BE: Belgium
BG: Bulgaria
CZ: Czechia
DK: Denmark
DE: Germany
EE: Estonia
IE: Ireland
EL: Greece
ES: Spain
FR: France
HR: Croatia
IT: Italy
CY: Cyprus
LV: Latvia
LT: Lithuania
LU: Luxembourg
HU: Hungary
MT: Malta
NL: Netherlands
AT: Austria
PL: Poland
PT: Portugal
RO: Romania
SI: Slovenia
SK: Slovakia
FI: Finland
SE: Sweden
INT_EU27_2020: Intra-EU27 (from 2020)
INT_EU28: Intra-EU28 (2013-2020)
INT_EU27_2007: Intra-EU27 (2007-2013)
EU27_2020_FOR: EU27 countri

### Source Market Extraction Definition

The source-market dataset provides annual overnight stays in Germany by tourists' country of residence.

The primary extraction uses:

- Destination: Germany
- Frequency: Annual
- Unit: Number of nights
- Accommodation scope: I551-I553
- Period: 2021-2025
- Residence: All available source-market categories

All residence categories will initially be preserved in the raw extraction.

Aggregate geographic categories will not be combined with individual source markets when calculating rankings or market shares because this would result in double counting.

Source-market classification and analytical filtering will be performed during data cleaning.

In [52]:
source_market_params = {
    "lang": "en",
    "geo": "DE",
    "freq": "A",
    "unit": "NR",
    "nace_r2": "I551-I553"
}

source_market_response = requests.get(
    source_market_url,
    params=source_market_params,
    timeout=30
)

print(
    "Status code:",
    source_market_response.status_code
)

germany_source_data = source_market_response.json()

print("Dataset:", germany_source_data.get("label"))
print("Dimensions:", germany_source_data.get("id"))
print("Sizes:", germany_source_data.get("size"))

Status code: 200
Dataset: Nights spent at tourist accommodation establishments  by country of origin of the tourist
Dimensions: ['freq', 'unit', 'nace_r2', 'c_resid', 'geo', 'time']
Sizes: [1, 1, 1, 63, 1, 36]


In [53]:
residence_index = (
    germany_source_data["dimension"]["c_resid"]
    ["category"]["index"]
)

residence_labels = (
    germany_source_data["dimension"]["c_resid"]
    ["category"]["label"]
)

time_index = (
    germany_source_data["dimension"]["time"]
    ["category"]["index"]
)

values = germany_source_data.get("value", {})
status = germany_source_data.get("status", {})

print("Residence categories:", len(residence_index))
print("Time periods:", len(time_index))
print("Numeric observations:", len(values))

Residence categories: 63
Time periods: 36
Numeric observations: 1701


In [54]:
source_market_records = []

n_time = len(time_index)

for residence_code, residence_position in residence_index.items():

    residence_name = residence_labels.get(
        residence_code,
        residence_code
    )

    for period, time_position in time_index.items():

        year = int(period)

        if 2021 <= year <= 2025:

            position = (
                residence_position * n_time
                + time_position
            )

            key = str(position)

            source_market_records.append({
                "destination_code": "DE",
                "destination": "Germany",
                "source_market_code": residence_code,
                "source_market": residence_name,
                "year": year,
                "foreign_nights": values.get(
                    key,
                    np.nan
                ),
                "status": status.get(
                    key,
                    None
                )
            })

germany_source_markets_raw = pd.DataFrame(
    source_market_records
)

print("Shape:", germany_source_markets_raw.shape)

germany_source_markets_raw.head(10)

Shape: (315, 7)


,destination_code,destination,source_market_code,source_market,year,foreign_nights,status
0,DE,Germany,EUR,Europe,2021,261386681.0,None
1,DE,Germany,EUR,Europe,2022,386328715.0,None
2,DE,Germany,EUR,Europe,2023,412829070.0,None
3,DE,Germany,EUR,Europe,2024,419240031.0,None
4,DE,Germany,EUR,Europe,2025,NaN,None
5,DE,Germany,BE,Belgium,2021,1394320.0,None
6,DE,Germany,BE,Belgium,2022,2617268.0,None
7,DE,Germany,BE,Belgium,2023,2939910.0,None
8,DE,Germany,BE,Belgium,2024,3044527.0,None
9,DE,Germany,BE,Belgium,2025,NaN,None


In [55]:
print(
    "Rows:",
    len(germany_source_markets_raw)
)

print(
    "Source-market categories:",
    germany_source_markets_raw[
        "source_market_code"
    ].nunique()
)

print(
    "Years:",
    sorted(
        germany_source_markets_raw["year"]
        .unique()
    )
)

print(
    "Missing observations:",
    germany_source_markets_raw[
        "foreign_nights"
    ].isna().sum()
)

print(
    "Duplicate source-year rows:",
    germany_source_markets_raw.duplicated(
        subset=["source_market_code", "year"]
    ).sum()
)

Rows: 315
Source-market categories: 63
Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Missing observations: 67
Duplicate source-year rows: 0


In [56]:
source_market_year_coverage = (
    germany_source_markets_raw
    .groupby("year")
    .agg(
        source_categories=(
            "source_market_code",
            "nunique"
        ),
        observations=(
            "foreign_nights",
            "count"
        ),
        missing_values=(
            "foreign_nights",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

source_market_year_coverage

,year,source_categories,observations,missing_values
0,2021,63,62,1
1,2022,63,62,1
2,2023,63,62,1
3,2024,63,62,1
4,2025,63,0,63


In [57]:
source_market_missing = (
    germany_source_markets_raw[
        germany_source_markets_raw[
            "foreign_nights"
        ].isna()
    ]
    .sort_values(
        ["year", "source_market_code"]
    )
)

source_market_missing

,destination_code,destination,source_market_code,source_market,year,foreign_nights,status
150,DE,Germany,INT_EU27_2007,Intra-EU27 (2007-2013),2021,NaN,None
151,DE,Germany,INT_EU27_2007,Intra-EU27 (2007-2013),2022,NaN,None
152,DE,Germany,INT_EU27_2007,Intra-EU27 (2007-2013),2023,NaN,None
153,DE,Germany,INT_EU27_2007,Intra-EU27 (2007-2013),2024,NaN,None
209,DE,Germany,AFR,Africa,2025,NaN,None
219,DE,Germany,AFR_OTH,Other African countries (aggregate changing ac...,2025,NaN,None
224,DE,Germany,AME,America,2025,NaN,None
249,DE,Germany,AME_C_S,Central and South America,2025,NaN,None
259,DE,Germany,AME_C_S_OTH,Other Central or South American countries,2025,NaN,None
229,DE,Germany,AME_N,Northern America,2025,NaN,None


### Source Market Data Availability

The annual Germany source-market extraction contains 63 residence categories for each year from 2021 through 2025.

Data validation identified an important availability boundary:

- 2021-2024 contain 62 numeric observations per year.
- The unavailable category during 2021-2024 is `INT_EU27_2007`, a historical aggregate that is not required for individual source-market analysis.
- 2025 currently contains no numeric observations for any residence category.

Therefore, the current official source-market analysis period will be **2021-2024**.

The unavailable 2025 observations will remain missing and will not be imputed or reconstructed.

This does not change the overall project period. Other tourism indicators contain complete 2025 data, while source-market analysis will use the latest complete period available for that specific dataset.

Aggregate residence categories will be preserved in the raw data but separated from individual source markets during data cleaning to prevent double counting.

In [58]:
source_market_file = (
    raw_path /
    "eurostat_germany_source_market_nights_annual_raw.csv"
)

germany_source_markets_raw.to_csv(
    source_market_file,
    index=False
)

print("Saved:", source_market_file)
print("Shape:", germany_source_markets_raw.shape)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/eurostat_germany_source_market_nights_annual_raw.csv
Shape: (315, 7)


In [59]:
source_market_metadata = pd.DataFrame({
    "dataset_name": [
        "Nights spent at tourist accommodation establishments by country of origin of the tourist"
    ],
    "dataset_code": [
        "tour_occ_ninraw"
    ],
    "source": [
        "Eurostat"
    ],
    "geographic_coverage": [
        "Germany by source market"
    ],
    "time_coverage": [
        "2021-2025"
    ],
    "frequency": [
        "Annual"
    ],
    "unit": [
        "Number of nights"
    ],
    "residence": [
        "Individual countries and geographic aggregates"
    ],
    "accommodation_scope": [
        "I551-I553"
    ],
    "collection_date": [
        datetime.now().date().isoformat()
    ],
    "analytical_use": [
        "Source market ranking, market share, growth, and strategic opportunity analysis"
    ],
    "notes": [
        "2021-2024 contain source-market observations. 2025 is currently unavailable for all residence categories. Aggregate categories require separation from individual markets before analysis."
    ]
})

collection_metadata = pd.concat(
    [collection_metadata, source_market_metadata],
    ignore_index=True
)

collection_metadata.to_csv(
    metadata_file,
    index=False
)

collection_metadata

,dataset_name,dataset_code,source,geographic_coverage,time_coverage,frequency,unit,residence,accommodation_scope,collection_date,analytical_use,notes
0,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"International tourism demand, seasonality, and...",2026-07 period available but numeric value mis...
1,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"EU benchmarking, growth, seasonality, statisti...",Common 2026 EU27 comparison window is January-...
2,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"EU benchmarking, tourism volume, growth, seaso...",Missing observations vary by country. Germany ...
3,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"Germany tourism performance, seasonality, 2026...",Complete 2021-2025. January-June 2026 availabl...
4,Nights spent at tourist accommodation establis...,tour_occ_ninraw,Eurostat,Germany by source market,2021-2025,Annual,Number of nights,Individual countries and geographic aggregates,I551-I553,2026-09-07,"Source market ranking, market share, growth, a...",2021-2024 contain source-market observations. ...


## 11. Source Market Arrivals

Source-market arrivals complement source-market overnight stays by measuring how many visitors from each country of residence arrive at tourist accommodation establishments in Germany.

Together, arrivals and nights allow calculation of average length of stay by source market:

Average Length of Stay = Source Market Nights / Source Market Arrivals

This supports a more strategic interpretation of source markets by distinguishing:

- high-volume markets
- long-stay markets
- fast-growing markets
- markets with relatively short stays
- markets that may offer greater tourism value despite lower arrival volume

The same geographic and accommodation definitions used for source-market nights will be applied where available.

In [60]:
source_arrivals_dataset_code = "tour_occ_arnraw"

source_arrivals_url = (
    f"{base_url}/{source_arrivals_dataset_code}"
)

source_arrivals_params_de = {
    "lang": "en",
    "geo": "DE"
}

source_arrivals_response = requests.get(
    source_arrivals_url,
    params=source_arrivals_params_de,
    timeout=30
)

print("Status code:", source_arrivals_response.status_code)
print("Request URL:", source_arrivals_response.url)

Status code: 200
Request URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tour_occ_arnraw?lang=en&geo=DE


In [61]:
source_arrivals_data = source_arrivals_response.json()

print("Dataset label:")
print(source_arrivals_data.get("label"))

print("\nDimensions:")
print(source_arrivals_data.get("id"))

print("\nDimension sizes:")
print(source_arrivals_data.get("size"))

Dataset label:
Arrivals at tourist accommodation establishments by country of origin of the tourist

Dimensions:
['freq', 'unit', 'c_resid', 'nace_r2', 'geo', 'time']

Dimension sizes:
[1, 2, 63, 5, 1, 36]


In [62]:
for dimension in source_arrivals_data.get("id", []):

    if dimension == "time":
        continue

    dim_data = (
        source_arrivals_data["dimension"][dimension]
        ["category"]
    )

    print(f"\n--- {dimension.upper()} ---")

    labels = dim_data.get("label", {})

    for code, label in labels.items():
        print(f"{code}: {label}")


--- FREQ ---
A: Annual

--- UNIT ---
NR: Number
PCH_PRE: Percentage change on previous period

--- C_RESID ---
EUR: Europe
BE: Belgium
BG: Bulgaria
CZ: Czechia
DK: Denmark
DE: Germany
EE: Estonia
IE: Ireland
EL: Greece
ES: Spain
FR: France
HR: Croatia
IT: Italy
CY: Cyprus
LV: Latvia
LT: Lithuania
LU: Luxembourg
HU: Hungary
MT: Malta
NL: Netherlands
AT: Austria
PL: Poland
PT: Portugal
RO: Romania
SI: Slovenia
SK: Slovakia
FI: Finland
SE: Sweden
INT_EU27_2020: Intra-EU27 (from 2020)
INT_EU28: Intra-EU28 (2013-2020)
INT_EU27_2007: Intra-EU27 (2007-2013)
EU27_2020_FOR: EU27 countries (from 2020) except reporting country
EFTA: European Free Trade Association
IS: Iceland
CH_LI: Switzerland and Liechtenstein
NO: Norway
UK: United Kingdom
TR: Türkiye
UA: Ukraine
RU: Russia
EUR_OTH: Other European countries (aggregate changing according to the context)
AFR: Africa
ZA: South Africa
AFR_OTH: Other African countries (aggregate changing according to the context)
AME: America
AME_N: Northern Americ

In [63]:
source_arrivals_params = {
    "lang": "en",
    "geo": "DE",
    "freq": "A",
    "unit": "NR",
    "nace_r2": "I551-I553"
}

source_arrivals_response = requests.get(
    source_arrivals_url,
    params=source_arrivals_params,
    timeout=30
)

print(
    "Status code:",
    source_arrivals_response.status_code
)

germany_source_arrivals_data = (
    source_arrivals_response.json()
)

print(
    "Dataset:",
    germany_source_arrivals_data.get("label")
)

print(
    "Dimensions:",
    germany_source_arrivals_data.get("id")
)

print(
    "Sizes:",
    germany_source_arrivals_data.get("size")
)

Status code: 200
Dataset: Arrivals at tourist accommodation establishments by country of origin of the tourist
Dimensions: ['freq', 'unit', 'c_resid', 'nace_r2', 'geo', 'time']
Sizes: [1, 1, 63, 1, 1, 36]


In [64]:
residence_index = (
    germany_source_arrivals_data[
        "dimension"
    ]["c_resid"]["category"]["index"]
)

residence_labels = (
    germany_source_arrivals_data[
        "dimension"
    ]["c_resid"]["category"]["label"]
)

time_index = (
    germany_source_arrivals_data[
        "dimension"
    ]["time"]["category"]["index"]
)

values = germany_source_arrivals_data.get(
    "value",
    {}
)

status = germany_source_arrivals_data.get(
    "status",
    {}
)

print(
    "Residence categories:",
    len(residence_index)
)

print(
    "Time periods:",
    len(time_index)
)

print(
    "Numeric observations:",
    len(values)
)

Residence categories: 63
Time periods: 36
Numeric observations: 1700


In [65]:
source_arrival_records = []

n_time = len(time_index)

for residence_code, residence_position in (
    residence_index.items()
):

    residence_name = residence_labels.get(
        residence_code,
        residence_code
    )

    for period, time_position in time_index.items():

        year = int(period)

        if 2021 <= year <= 2025:

            position = (
                residence_position * n_time
                + time_position
            )

            key = str(position)

            source_arrival_records.append({
                "destination_code": "DE",
                "destination": "Germany",
                "source_market_code":
                    residence_code,
                "source_market":
                    residence_name,
                "year": year,
                "foreign_arrivals":
                    values.get(key, np.nan),
                "status":
                    status.get(key, None)
            })

germany_source_arrivals_raw = (
    pd.DataFrame(source_arrival_records)
)

print(
    "Shape:",
    germany_source_arrivals_raw.shape
)

germany_source_arrivals_raw.head(10)

Shape: (315, 7)


,destination_code,destination,source_market_code,source_market,year,foreign_arrivals,status
0,DE,Germany,EUR,Europe,2021,92065343.0,None
1,DE,Germany,EUR,Europe,2022,152891215.0,None
2,DE,Germany,EUR,Europe,2023,170315670.0,None
3,DE,Germany,EUR,Europe,2024,175758213.0,None
4,DE,Germany,EUR,Europe,2025,NaN,None
5,DE,Germany,BE,Belgium,2021,621993.0,None
6,DE,Germany,BE,Belgium,2022,1216635.0,None
7,DE,Germany,BE,Belgium,2023,1396964.0,None
8,DE,Germany,BE,Belgium,2024,1465552.0,None
9,DE,Germany,BE,Belgium,2025,NaN,None


In [66]:
print(
    "Rows:",
    len(germany_source_arrivals_raw)
)

print(
    "Source-market categories:",
    germany_source_arrivals_raw[
        "source_market_code"
    ].nunique()
)

print(
    "Years:",
    sorted(
        germany_source_arrivals_raw[
            "year"
        ].unique()
    )
)

print(
    "Missing observations:",
    germany_source_arrivals_raw[
        "foreign_arrivals"
    ].isna().sum()
)

print(
    "Duplicate source-year rows:",
    germany_source_arrivals_raw.duplicated(
        subset=[
            "source_market_code",
            "year"
        ]
    ).sum()
)

Rows: 315
Source-market categories: 63
Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Missing observations: 67
Duplicate source-year rows: 0


In [67]:
source_arrivals_year_coverage = (
    germany_source_arrivals_raw
    .groupby("year")
    .agg(
        source_categories=(
            "source_market_code",
            "nunique"
        ),
        observations=(
            "foreign_arrivals",
            "count"
        ),
        missing_values=(
            "foreign_arrivals",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

source_arrivals_year_coverage

,year,source_categories,observations,missing_values
0,2021,63,62,1
1,2022,63,62,1
2,2023,63,62,1
3,2024,63,62,1
4,2025,63,0,63


In [68]:
source_arrivals_missing = (
    germany_source_arrivals_raw[
        germany_source_arrivals_raw[
            "foreign_arrivals"
        ].isna()
    ]
    .sort_values(
        ["year", "source_market_code"]
    )
)

source_arrivals_missing

,destination_code,destination,source_market_code,source_market,year,foreign_arrivals,status
150,DE,Germany,INT_EU27_2007,Intra-EU27 (2007-2013),2021,NaN,None
151,DE,Germany,INT_EU27_2007,Intra-EU27 (2007-2013),2022,NaN,None
152,DE,Germany,INT_EU27_2007,Intra-EU27 (2007-2013),2023,NaN,None
153,DE,Germany,INT_EU27_2007,Intra-EU27 (2007-2013),2024,NaN,None
209,DE,Germany,AFR,Africa,2025,NaN,None
219,DE,Germany,AFR_OTH,Other African countries (aggregate changing ac...,2025,NaN,None
224,DE,Germany,AME,America,2025,NaN,None
249,DE,Germany,AME_C_S,Central and South America,2025,NaN,None
259,DE,Germany,AME_C_S_OTH,Other Central or South American countries,2025,NaN,None
229,DE,Germany,AME_N,Northern America,2025,NaN,None


### Source Market Arrivals Data Validation

The Germany source-market arrivals extraction contains 63 residence categories for each year from 2021 through 2025.

Data availability matches the source-market overnight stays dataset:

- 2021-2024 contain 62 numeric observations per year.
- The unavailable category during 2021-2024 is `INT_EU27_2007`, a historical aggregate that is not required for individual source-market analysis.
- 2025 currently contains no numeric observations for any residence category.
- No duplicate source-market-year records were identified.

Therefore, the matched source-market analysis period for arrivals and overnight stays is **2021-2024**.

This allows source-market arrivals, overnight stays, growth, market share, and average length of stay to be analyzed using consistent periods and definitions.

The unavailable 2025 observations will remain missing and will not be imputed.

In [69]:
source_arrivals_file = (
    raw_path /
    "eurostat_germany_source_market_arrivals_annual_raw.csv"
)

germany_source_arrivals_raw.to_csv(
    source_arrivals_file,
    index=False
)

print("Saved:", source_arrivals_file)
print("Shape:", germany_source_arrivals_raw.shape)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/eurostat_germany_source_market_arrivals_annual_raw.csv
Shape: (315, 7)


In [70]:
source_arrivals_metadata = pd.DataFrame({
    "dataset_name": [
        "Arrivals at tourist accommodation establishments by country of origin of the tourist"
    ],
    "dataset_code": [
        "tour_occ_arnraw"
    ],
    "source": [
        "Eurostat"
    ],
    "geographic_coverage": [
        "Germany by source market"
    ],
    "time_coverage": [
        "2021-2025"
    ],
    "frequency": [
        "Annual"
    ],
    "unit": [
        "Number of arrivals"
    ],
    "residence": [
        "Individual countries and geographic aggregates"
    ],
    "accommodation_scope": [
        "I551-I553"
    ],
    "collection_date": [
        datetime.now().date().isoformat()
    ],
    "analytical_use": [
        "Source market volume, growth, average length of stay, and strategic opportunity analysis"
    ],
    "notes": [
        "2021-2024 contain source-market observations. 2025 is currently unavailable for all residence categories. Aggregate categories require separation from individual markets before analysis."
    ]
})

collection_metadata = pd.concat(
    [
        collection_metadata,
        source_arrivals_metadata
    ],
    ignore_index=True
)

collection_metadata.to_csv(
    metadata_file,
    index=False
)

collection_metadata

,dataset_name,dataset_code,source,geographic_coverage,time_coverage,frequency,unit,residence,accommodation_scope,collection_date,analytical_use,notes
0,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"International tourism demand, seasonality, and...",2026-07 period available but numeric value mis...
1,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"EU benchmarking, growth, seasonality, statisti...",Common 2026 EU27 comparison window is January-...
2,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"EU benchmarking, tourism volume, growth, seaso...",Missing observations vary by country. Germany ...
3,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"Germany tourism performance, seasonality, 2026...",Complete 2021-2025. January-June 2026 availabl...
4,Nights spent at tourist accommodation establis...,tour_occ_ninraw,Eurostat,Germany by source market,2021-2025,Annual,Number of nights,Individual countries and geographic aggregates,I551-I553,2026-09-07,"Source market ranking, market share, growth, a...",2021-2024 contain source-market observations. ...
5,Arrivals at tourist accommodation establishmen...,tour_occ_arnraw,Eurostat,Germany by source market,2021-2025,Annual,Number of arrivals,Individual countries and geographic aggregates,I551-I553,2026-09-07,"Source market volume, growth, average length o...",2021-2024 contain source-market observations. ...


## 12. Economic Performance Data

Tourism competitiveness is not determined only by visitor volume. Economic value is also important.

For this project, international travel receipts and expenditure will be collected using Balance of Payments data.

The Balance of Payments travel item distinguishes:

- Travel credits: expenditure by non-resident travellers in the reporting economy
- Travel debits: expenditure by residents while travelling abroad

Travel credits will be used as the primary inbound tourism economic indicator.

Travel debits may be retained for contextual comparison but will not be interpreted as inbound tourism revenue.

These indicators are conceptually different from accommodation arrivals and overnight stays. They measure international travel expenditure rather than activity recorded by tourist accommodation establishments.

In [71]:
economic_dataset_code = "bop_its6_det"

economic_url = (
    f"{base_url}/{economic_dataset_code}"
)

economic_params_de = {
    "lang": "en",
    "geo": "DE"
}

economic_response = requests.get(
    economic_url,
    params=economic_params_de,
    timeout=30
)

print("Status code:", economic_response.status_code)
print("Request URL:", economic_response.url)

Status code: 200
Request URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/bop_its6_det?lang=en&geo=DE


In [72]:
economic_data = economic_response.json()

print("Dataset label:")
print(economic_data.get("label"))

print("\nDimensions:")
print(economic_data.get("id"))

print("\nDimension sizes:")
print(economic_data.get("size"))

Dataset label:
International trade in services (since 2010) (BPM6)

Dimensions:
['freq', 'currency', 'bop_item', 'stk_flow', 'partner', 'geo', 'time']

Dimension sizes:
[1, 1, 143, 3, 307, 1, 16]


In [73]:
for dimension in economic_data.get("id", []):

    if dimension == "time":
        continue

    dim_data = (
        economic_data["dimension"][dimension]
        ["category"]
    )

    print(f"\n--- {dimension.upper()} ---")

    labels = dim_data.get("label", {})

    for code, label in labels.items():
        print(f"{code}: {label}")


--- FREQ ---
A: Annual

--- CURRENCY ---
MIO_EUR: Million euro

--- BOP_ITEM ---
S: Services
SA: Services: manufacturing services on physical inputs owned by others
SAY: Services: goods for processing in reporting economy - goods returned, goods received
SAZ: Services: goods for processing abroad - goods sent, goods returned
SB: Services: maintenance and repair services n.i.e.
SC: Services: transport
SC1: Services: sea transport
SC11: Services: sea transport; passenger
SC12: Services: sea transport; freight
SC13: Services: sea transport; other than passenger and freight
SC2: Services: air transport
SC21: Services: air transport; passenger
SC22: Services: air transport; freight
SC23: Services: air transport; other than passenger and freight
SC3: Services: other modes of transport
SC31: Services: other modes of transport; passenger
SC32: Services: other modes of transport; freight
SC33: Services: other modes of transport; other than passenger and freight
SC3A: Services: space transport


In [74]:
travel_params_de = {
    "lang": "en",
    "geo": "DE",
    "freq": "A",
    "currency": "MIO_EUR",
    "bop_item": "SD"
}

travel_response_de = requests.get(
    economic_url,
    params=travel_params_de,
    timeout=30
)

print(
    "Status code:",
    travel_response_de.status_code
)

travel_data_de = travel_response_de.json()

print("Dataset:", travel_data_de.get("label"))
print("Dimensions:", travel_data_de.get("id"))
print("Sizes:", travel_data_de.get("size"))

Status code: 200
Dataset: International trade in services (since 2010) (BPM6)
Dimensions: ['freq', 'currency', 'bop_item', 'stk_flow', 'partner', 'geo', 'time']
Sizes: [1, 1, 1, 3, 307, 1, 16]


In [75]:
travel_time_labels = (
    travel_data_de["dimension"]["time"]
    ["category"]["label"]
)

print("Available periods:")

for code, label in travel_time_labels.items():
    print(code, ":", label)

Available periods:
2010 : 2010
2011 : 2011
2012 : 2012
2013 : 2013
2014 : 2014
2015 : 2015
2016 : 2016
2017 : 2017
2018 : 2018
2019 : 2019
2020 : 2020
2021 : 2021
2022 : 2022
2023 : 2023
2024 : 2024
2025 : 2025


In [76]:
partner_index = (
    travel_data_de["dimension"]["partner"]
    ["category"]["index"]
)

partner_labels = (
    travel_data_de["dimension"]["partner"]
    ["category"]["label"]
)

flow_index = (
    travel_data_de["dimension"]["stk_flow"]
    ["category"]["index"]
)

time_index = (
    travel_data_de["dimension"]["time"]
    ["category"]["index"]
)

values = travel_data_de.get("value", {})

print("Partner categories:", len(partner_index))
print("Flows:", len(flow_index))
print("Years:", len(time_index))
print("Numeric observations:", len(values))

Partner categories: 307
Flows: 3
Years: 16
Numeric observations: 3639


In [77]:
candidate_partners = [
    "EUR",
    "EU27_2020",
    "EXT_EU27_2020",
    "WRL_REST"
]

for code in candidate_partners:

    if code in partner_labels:
        print(
            code,
            ":",
            partner_labels[code]
        )

EUR : Europe
EU27_2020 : European Union - 27 countries (from 2020)
EXT_EU27_2020 : Extra-EU27 (from 2020)
WRL_REST : Rest of the world


In [78]:
for code, label in partner_labels.items():

    label_lower = label.lower()

    if (
        "world" in label_lower
        or "total" in label_lower
        or "rest" in label_lower
    ):
        print(code, ":", label)

WRL_REST : Rest of the world


In [79]:
economic_partner_params = {
    "lang": "en",
    "geo": "DE",
    "freq": "A",
    "currency": "MIO_EUR",
    "bop_item": "SD",
    "partner": [
        "EU27_2020",
        "EXT_EU27_2020"
    ]
}

In [80]:
economic_records = []

for partner_code in [
    "EU27_2020",
    "EXT_EU27_2020"
]:

    params = {
        "lang": "en",
        "geo": "DE",
        "freq": "A",
        "currency": "MIO_EUR",
        "bop_item": "SD",
        "partner": partner_code
    }

    response = requests.get(
        economic_url,
        params=params,
        timeout=30
    )

    print(
        partner_code,
        response.status_code
    )

    partner_data = response.json()

    print(
        partner_data.get("label")
    )

    print(
        partner_data.get("id")
    )

    print(
        partner_data.get("size")
    )

EU27_2020 200
International trade in services (since 2010) (BPM6)
['freq', 'currency', 'bop_item', 'stk_flow', 'partner', 'geo', 'time']
[1, 1, 1, 3, 1, 1, 16]
EXT_EU27_2020 200
International trade in services (since 2010) (BPM6)
['freq', 'currency', 'bop_item', 'stk_flow', 'partner', 'geo', 'time']
[1, 1, 1, 3, 1, 1, 16]


In [81]:
economic_records = []

partner_codes = {
    "EU27_2020": "EU27",
    "EXT_EU27_2020": "Extra-EU27"
}

for partner_code, partner_name in (
    partner_codes.items()
):

    params = {
        "lang": "en",
        "geo": "DE",
        "freq": "A",
        "currency": "MIO_EUR",
        "bop_item": "SD",
        "partner": partner_code
    }

    response = requests.get(
        economic_url,
        params=params,
        timeout=30
    )

    partner_data = response.json()

    flow_index = (
        partner_data["dimension"]["stk_flow"]
        ["category"]["index"]
    )

    time_index = (
        partner_data["dimension"]["time"]
        ["category"]["index"]
    )

    values = partner_data.get(
        "value",
        {}
    )

    status = partner_data.get(
        "status",
        {}
    )

    n_time = len(time_index)

    for flow_code, flow_position in (
        flow_index.items()
    ):

        for year_code, time_position in (
            time_index.items()
        ):

            year = int(year_code)

            if 2021 <= year <= 2025:

                position = (
                    flow_position * n_time
                    + time_position
                )

                key = str(position)

                economic_records.append({
                    "country_code": "DE",
                    "country": "Germany",
                    "partner_code":
                        partner_code,
                    "partner":
                        partner_name,
                    "flow_code":
                        flow_code,
                    "year":
                        year,
                    "travel_mio_eur":
                        values.get(
                            key,
                            np.nan
                        ),
                    "status":
                        status.get(
                            key,
                            None
                        )
                })

germany_travel_economic_raw = (
    pd.DataFrame(economic_records)
)

germany_travel_economic_raw

,country_code,country,partner_code,partner,flow_code,year,travel_mio_eur,status
0,DE,Germany,EU27_2020,EU27,CRE,2021,13317.0,NaN
1,DE,Germany,EU27_2020,EU27,CRE,2022,20539.0,NaN
2,DE,Germany,EU27_2020,EU27,CRE,2023,23223.0,NaN
3,DE,Germany,EU27_2020,EU27,CRE,2024,24860.0,NaN
4,DE,Germany,EU27_2020,EU27,CRE,2025,25711.0,p
5,DE,Germany,EU27_2020,EU27,DEB,2021,34061.0,NaN
6,DE,Germany,EU27_2020,EU27,DEB,2022,60740.0,NaN
7,DE,Germany,EU27_2020,EU27,DEB,2023,73692.0,NaN
8,DE,Germany,EU27_2020,EU27,DEB,2024,71236.0,NaN
9,DE,Germany,EU27_2020,EU27,DEB,2025,75502.0,p


In [82]:
print(
    "Rows:",
    len(germany_travel_economic_raw)
)

print(
    "Missing values:",
    germany_travel_economic_raw[
        "travel_mio_eur"
    ].isna().sum()
)

print(
    "Duplicates:",
    germany_travel_economic_raw.duplicated(
        subset=[
            "partner_code",
            "flow_code",
            "year"
        ]
    ).sum()
)

Rows: 30
Missing values: 0
Duplicates: 0


In [83]:
economic_coverage = (
    germany_travel_economic_raw
    .groupby(
        ["partner", "flow_code"]
    )
    .agg(
        first_year=(
            "year",
            "min"
        ),
        last_year=(
            "year",
            "max"
        ),
        observations=(
            "travel_mio_eur",
            "count"
        ),
        missing_values=(
            "travel_mio_eur",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

economic_coverage

,partner,flow_code,first_year,last_year,observations,missing_values
0,EU27,BAL,2021,2025,5,0
1,EU27,CRE,2021,2025,5,0
2,EU27,DEB,2021,2025,5,0
3,Extra-EU27,BAL,2021,2025,5,0
4,Extra-EU27,CRE,2021,2025,5,0
5,Extra-EU27,DEB,2021,2025,5,0


In [84]:
germany_travel_economic_raw.pivot_table(
    index="year",
    columns=[
        "partner",
        "flow_code"
    ],
    values="travel_mio_eur",
    aggfunc="first"
)

partner       EU27                   Extra-EU27                  
flow_code      BAL      CRE      DEB        BAL      CRE      DEB
year                                                             
2021      -20744.0  13317.0  34061.0    -3556.0   5510.0   9065.0
2022      -40201.0  20539.0  60740.0   -14561.0   9718.0  24279.0
2023      -50469.0  23223.0  73692.0   -21181.0  11769.0  32950.0
2024      -46376.0  24860.0  71236.0   -23391.0  12195.0  35586.0
2025      -49791.0  25711.0  75502.0   -27031.0  12061.0  39092.0

### Germany Travel Economic Data Validation

The Germany Balance of Payments travel dataset contains complete annual observations from 2021 through 2025 for:

- EU27 partner countries
- Extra-EU27 partner countries
- Credits
- Debits
- Balance

A total of 30 observations were extracted, with no missing values and no duplicate partner-flow-year combinations.

The 2025 observations are flagged `p` by Eurostat, indicating provisional values. These observations will be retained but clearly distinguished from final historical data.

For economic analysis:

- Travel credits represent international travel receipts.
- Travel debits represent international travel expenditure by German residents abroad.
- The travel balance represents credits minus debits.
- EU27 and Extra-EU27 partner aggregates will be retained separately in the raw data.

Combined global indicators may later be derived from the mutually exclusive EU27 and Extra-EU27 aggregates during data processing, rather than modifying the raw source data.

In [85]:
economic_file = (
    raw_path /
    "eurostat_germany_travel_bop_annual_raw.csv"
)

germany_travel_economic_raw.to_csv(
    economic_file,
    index=False
)

print("Saved:", economic_file)
print("Shape:", germany_travel_economic_raw.shape)

Saved: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw/eurostat_germany_travel_bop_annual_raw.csv
Shape: (30, 8)


In [86]:
economic_metadata = pd.DataFrame({
    "dataset_name": [
        "International trade in services (since 2010) (BPM6)"
    ],
    "dataset_code": [
        "bop_its6_det"
    ],
    "source": [
        "Eurostat"
    ],
    "geographic_coverage": [
        "Germany with EU27 and Extra-EU27 partners"
    ],
    "time_coverage": [
        "2021-2025"
    ],
    "frequency": [
        "Annual"
    ],
    "unit": [
        "Million euro"
    ],
    "residence": [
        "Not applicable - Balance of Payments partner geography"
    ],
    "accommodation_scope": [
        "Not applicable"
    ],
    "collection_date": [
        datetime.now().date().isoformat()
    ],
    "analytical_use": [
        "International travel receipts, expenditure, balance, economic performance, and tourism competitiveness"
    ],
    "notes": [
        "Travel item SD. EU27 and Extra-EU27 partner aggregates collected separately. Credits represent travel receipts and debits represent travel expenditure. All 2025 observations are provisional."
    ]
})

collection_metadata = pd.concat(
    [
        collection_metadata,
        economic_metadata
    ],
    ignore_index=True
)

collection_metadata.to_csv(
    metadata_file,
    index=False
)

collection_metadata

,dataset_name,dataset_code,source,geographic_coverage,time_coverage,frequency,unit,residence,accommodation_scope,collection_date,analytical_use,notes
0,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"International tourism demand, seasonality, and...",2026-07 period available but numeric value mis...
1,Nights spent at tourist accommodation establis...,tour_occ_nim,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of nights,Foreign country,I551-I553,2026-09-07,"EU benchmarking, growth, seasonality, statisti...",Common 2026 EU27 comparison window is January-...
2,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,EU27 member states,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"EU benchmarking, tourism volume, growth, seaso...",Missing observations vary by country. Germany ...
3,Arrivals at tourist accommodation establishmen...,tour_occ_arm,Eurostat,Germany,2021-01 to 2026-07,Monthly,Number of arrivals,Foreign country,I551-I553,2026-09-07,"Germany tourism performance, seasonality, 2026...",Complete 2021-2025. January-June 2026 availabl...
4,Nights spent at tourist accommodation establis...,tour_occ_ninraw,Eurostat,Germany by source market,2021-2025,Annual,Number of nights,Individual countries and geographic aggregates,I551-I553,2026-09-07,"Source market ranking, market share, growth, a...",2021-2024 contain source-market observations. ...
5,Arrivals at tourist accommodation establishmen...,tour_occ_arnraw,Eurostat,Germany by source market,2021-2025,Annual,Number of arrivals,Individual countries and geographic aggregates,I551-I553,2026-09-07,"Source market volume, growth, average length o...",2021-2024 contain source-market observations. ...
6,International trade in services (since 2010) (...,bop_its6_det,Eurostat,Germany with EU27 and Extra-EU27 partners,2021-2025,Annual,Million euro,Not applicable - Balance of Payments partner g...,Not applicable,2026-09-07,"International travel receipts, expenditure, ba...",Travel item SD. EU27 and Extra-EU27 partner ag...


## 13. Data Collection Summary

The data collection stage assembled the core official datasets required to evaluate Germany's tourism competitiveness.

Seven documented data extractions were collected from Eurostat:

1. Germany monthly foreign overnight stays
2. EU27 monthly foreign overnight stays
3. EU27 monthly foreign arrivals
4. Germany monthly foreign arrivals
5. Germany annual source-market overnight stays
6. Germany annual source-market arrivals
7. Germany annual Balance of Payments travel data

Together, these datasets support analysis of tourism volume, growth, EU competitive position, source markets, seasonality, average length of stay, economic performance, and 2026 year-to-date performance.

### Data Availability Boundaries

The datasets do not share identical reporting periods.

- Germany monthly arrivals and overnight stays are complete through June 2026 for the current year-to-date analysis.
- EU27 monthly availability varies by country, requiring period-completeness checks before comparative calculations.
- Some historical observations are confidential or unavailable and will remain missing.
- Germany source-market arrivals and overnight stays are analytically available through 2024. The 2025 source-market observations are currently unavailable.
- Balance of Payments travel data are available through 2025, with 2025 observations identified as provisional.

These differences will be preserved rather than artificially harmonized through imputation.

### Data Integrity Principles

The raw datasets will remain unchanged in `data/raw/`.

The next stage will:

- standardize data types and column structures
- preserve missing values and Eurostat status flags
- distinguish confidential, provisional, and unavailable observations
- classify individual source markets separately from geographic aggregates
- identify complete country-year and country-period observations
- validate keys and duplicates
- create analysis-ready datasets without modifying the original raw files

Derived indicators such as annual totals, growth rates, market shares, average length of stay, rankings, and combined economic indicators will be created only after the underlying observations have passed data-quality and comparability checks.

In [87]:
raw_files = sorted(
    file.name
    for file in raw_path.glob("*.csv")
)

print("Raw CSV files:", len(raw_files))

for file in raw_files:
    print("-", file)

Raw CSV files: 9
- data_collection_metadata.csv
- eu27_country_reference.csv
- eurostat_eu27_foreign_arrivals_monthly_raw.csv
- eurostat_eu27_foreign_nights_monthly_raw.csv
- eurostat_germany_foreign_arrivals_monthly_raw.csv
- eurostat_germany_foreign_nights_monthly_raw.csv
- eurostat_germany_source_market_arrivals_annual_raw.csv
- eurostat_germany_source_market_nights_annual_raw.csv
- eurostat_germany_travel_bop_annual_raw.csv
